In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().absolute().parent))


In [2]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    LearningRateMonitor,
)
from torch.nn import MSELoss
from torch.optim import Adam
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import seaborn as sns

import pandas as pd
import geopandas as gpd
from pathlib import Path


from src.data_models.caravanify import Caravanify, CaravanifyConfig

from src.data_models.datamodule import HydroDataModule

from sklearn.pipeline import Pipeline

from src.preprocessing.grouped import GroupedTransformer
# from src.preprocessing.log_scale import LogTransformer
from src.preprocessing.standard_scale import StandardScaleTransformer

from src.model_evaluation.evaluators import TSForecastEvaluator
from src.models.ealstm import EALSTMConfig, LitEALSTM

---

In [3]:
CA_config = CaravanifyConfig(
    attributes_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/attributes",
    timeseries_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/timeseries/csv",
    shapefile_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/shapefiles",
    human_influence_path="/Users/cooper/Desktop/CAMELS-CH/src/human_influence_index/results/human_influence_classification.csv",
    gauge_id_prefix="CA",
    use_hydroatlas_attributes=True,
    use_caravan_attributes=True,
    use_other_attributes=True,
)

CH_config = CaravanifyConfig(
    attributes_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CH/post_processed/attributes",
    timeseries_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CH/post_processed/timeseries/csv",
    shapefile_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CH/post_processed/shapefiles",
    human_influence_path="/Users/cooper/Desktop/CAMELS-CH/src/human_influence_index/results/human_influence_classification.csv",
    gauge_id_prefix="CH",
    use_hydroatlas_attributes=True,
    use_caravan_attributes=True,
    use_other_attributes=True,
)

CL_config = CaravanifyConfig(
    attributes_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CL/post_processed/attributes",
    timeseries_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CL/post_processed/timeseries/csv",
    shapefile_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CL/post_processed/shapefiles",
    human_influence_path="/Users/cooper/Desktop/CAMELS-CH/src/human_influence_index/results/human_influence_classification.csv",
    gauge_id_prefix="CL",
    use_hydroatlas_attributes=True,
    use_caravan_attributes=True,
    use_other_attributes=True,
)

USA_config = CaravanifyConfig(
    attributes_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/USA/post_processed/attributes",
    timeseries_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/USA/post_processed/timeseries/csv",
    shapefile_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/USA/post_processed/shapefiles",
    human_influence_path="/Users/cooper/Desktop/CAMELS-CH/src/human_influence_index/results/human_influence_classification.csv",
    gauge_id_prefix="USA",
    use_hydroatlas_attributes=True,
    use_caravan_attributes=True,
    use_other_attributes=True,
)

In [4]:
CA_caravan = Caravanify(CA_config)
CH_caravan = Caravanify(CH_config)
CL_caravan = Caravanify(CL_config)
USA_caravan = Caravanify(USA_config)

CA_basins = CA_caravan.get_all_gauge_ids()
CH_basins = CH_caravan.get_all_gauge_ids()
CL_basins = CL_caravan.get_all_gauge_ids()
USA_basins = USA_caravan.get_all_gauge_ids()

CA_gdf = CA_caravan.get_shapefiles()
CH_gdf = CH_caravan.get_shapefiles()
CL_gdf = CL_caravan.get_shapefiles()
USA_gdf = USA_caravan.get_shapefiles()

# Concatenate all gdfs 
all_gdfs = [CA_gdf, CH_gdf, CL_gdf, USA_gdf]
combined_gdf = pd.concat(all_gdfs, ignore_index=True)



In [5]:
ids_CH = CH_caravan.get_all_gauge_ids()
ids_CA = CA_caravan.get_all_gauge_ids()
ids_CL = CL_caravan.get_all_gauge_ids()
ids_USA = USA_caravan.get_all_gauge_ids()

# Print the length of each list
print(f"Number of basins in CH: {len(ids_CH)}")
print(f"Number of basins in CA: {len(ids_CA)}")
print(f"Number of basins in CL: {len(ids_CL)}")
print(f"Number of basins in USA: {len(ids_USA)}")

Number of basins in CH: 135
Number of basins in CA: 78
Number of basins in CL: 505
Number of basins in USA: 671


In [8]:
CH_caravan.load_stations(ids_CH)
CA_caravan.load_stations(ids_CA)
CL_caravan.load_stations(ids_CL)
USA_caravan.load_stations(ids_USA)
ts_CH = CH_caravan.get_time_series()
ts_CA = CA_caravan.get_time_series()
ts_CL = CL_caravan.get_time_series()
ts_USA = USA_caravan.get_time_series()

In [9]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assume ts_CA, ts_CL, ts_USA are your time-series DataFrames.

# 1. Add a region column to each DataFrame
ts_CA['region'] = 'CA'
ts_CL['region'] = 'CL'
ts_USA['region'] = 'USA'
ts_CH['region'] = 'CH'

# 2. Ensure the date column is datetime and extract the year
for ts in [ts_CA, ts_CL, ts_USA, ts_CH]:
    ts['date'] = pd.to_datetime(ts['date'])
    ts['year'] = ts['date'].dt.year

# 3. Combine the DataFrames
ts_all = pd.concat([ts_CA, ts_CL, ts_USA, ts_CH], ignore_index=True)


# 4. Compute the temporal span (min and max dates) per region
temporal_span = ts_all.groupby('region')['date'].agg(['min', 'max'])
print("Temporal span per region:")
print(temporal_span)


Temporal span per region:
              min        max
region                      
CA     2000-01-02 2024-12-31
CH     1981-01-02 2020-12-31
CL     1951-01-01 2023-12-30
USA    1951-01-01 2023-12-30


In [ ]:
# Filter the GeoDataFrame to include only the basins of interest
CH_gdf = CH_gdf[CH_gdf["gauge_id"].isin(ids_CH)]

# Reproject the GeoDataFrame to a projected CRS (choose one appropriate for your region)
# For Switzerland, EPSG:2056 is a good choice. For a global example, you might use EPSG:3857.
CH_gdf_proj = CH_gdf.to_crs(epsg=2056)  # or use epsg=3857 if no local CRS is available

# Compute the area in square meters and convert to square kilometers
CH_gdf_proj["area_km2"] = CH_gdf_proj.geometry.area / 1e6

# Print the summary statistics
print("Area (km²) - Combined:")
print(f"Min: {CH_gdf_proj['area_km2'].min()}")
print(f"Max: {CH_gdf_proj['area_km2'].max()}")
print(f"Mean: {CH_gdf_proj['area_km2'].mean()}")
print(f"Std: {CH_gdf_proj['area_km2'].std()}")

In [ ]:
# Plor the plygons and color them by area
fig, ax = plt.subplots(figsize=(10, 10))
# Plot the polygons with a color map based on the area
CH_gdf_proj.plot(column='area_km2', cmap='viridis', legend=True, ax=ax)
# Set the title and labels
ax.set_title('Area of Polygons in km²')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
# Show the plot
plt.show()


---

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# Create figure and axes with a projection
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={"projection": ccrs.PlateCarree()})

# Add base map features
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.add_feature(cfeature.LAND, alpha=0.1)
ax.add_feature(cfeature.OCEAN, alpha=0.1)
ax.add_feature(cfeature.LAKES, alpha=0.2)
ax.add_feature(cfeature.RIVERS, alpha=0.2)

# Color map for different countries
colors = {"CA": "red", "CH": "blue", "CL": "green", "USA": "purple"}

# Plot each basin with color based on country prefix
for idx, row in combined_gdf.iterrows():
    country_prefix = (
        row["gauge_id"].split("_")[0] if "_" in row["gauge_id"] else "Unknown"
    )
    color = colors.get(country_prefix, "gray")
    ax.add_geometries(
        [row.geometry],
        crs=ccrs.PlateCarree(),
        facecolor=color,
        edgecolor=None,
        alpha=1,
    )

# Add legend
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor=color, edgecolor=None, alpha=1, label=country)
    for country, color in colors.items()
]
ax.legend(handles=legend_elements, loc="lower left")

for spine in ax.spines.values():
    spine.set_visible(False)
# Set title and adjust layout
# plt.title("Basin Shapefile Overview")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.patches import FancyArrowPatch



# Create figure and axes
fig, ax = plt.subplots(figsize=(10, 6), 
                     subplot_kw={'projection': ccrs.PlateCarree()})

# Add cartography features
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
ax.add_feature(cfeature.LAND, alpha=0.3)
ax.add_feature(cfeature.OCEAN, alpha=0.2)
ax.add_feature(cfeature.LAKES, alpha=0.3)

# Plot CA basins
CA_gdf.plot(ax=ax, color='red', edgecolor='#ba3c3e', 
          alpha=0.6, linewidth=0.5)

# Add country labels for Central Asia
ax.text(74.7661, 41.2044, 'Kyrgyzstan', transform=ccrs.PlateCarree(), 
       fontsize=10, fontweight='bold', ha='center')
ax.text(71.2761, 38.8610, 'Tajikistan', transform=ccrs.PlateCarree(), 
       fontsize=10, fontweight='bold', ha='center')
# ax.text(67.71, 33.9391, 'Afghanistan', transform=ccrs.PlateCarree(), 
#        fontsize=10, ha='center')
# ax.text(64.5853, 41.3775, 'Uzbekistan', transform=ccrs.PlateCarree(),
#        fontsize=10, ha='center')
# ax.text(59.5563, 38.9697, 'Turkmenistan', transform=ccrs.PlateCarree(),
#        fontsize=10, ha='center')

# Add gridlines with labels
gl = ax.gridlines(draw_labels=True, linewidth=0.5, 
                alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 10}
gl.ylabel_style = {'size': 10}

for spine in ax.spines.values():
    spine.set_visible(False)

# Set specific spines to be visible for axes
ax.spines['left'].set_visible(True)  # y-axis
ax.spines['bottom'].set_visible(True)  # x-axis

arrow_x, arrow_y = 68.5, 42  # Position it in the top right
arrow_length = 0.5

# Create the arrow
north_arrow = FancyArrowPatch(
    (arrow_x, arrow_y),
    (arrow_x, arrow_y + arrow_length),
    transform=ccrs.PlateCarree(),
    color='black',
    linewidth=1,
    arrowstyle='-|>',
    mutation_scale=15
)
ax.add_patch(north_arrow)

# Add "N" label
ax.text(arrow_x, arrow_y + arrow_length + 0.2, 'N', 
        transform=ccrs.PlateCarree(), 
        ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()